In [60]:
# import all necessary libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
import spacy # available on Google Colab
from sklearn.model_selection import train_test_split

# Dataset


In [61]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1. Data Preprocessing:
- Load the slogan dataset

In [62]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/slogan-valid.csv')


Inspecting the dataset structure

In [63]:
df.info()

df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5346 entries, 0 to 5345
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   desc           5346 non-null   object
 1   output         5346 non-null   object
 2   type           5346 non-null   object
 3   company        5346 non-null   object
 4   industry       5346 non-null   object
 5   url            5346 non-null   object
 6   alias          4017 non-null   object
 7   desc_masked    5346 non-null   object
 8   output_masked  5346 non-null   object
 9   ent_dict       5346 non-null   object
 10  unsupported    5346 non-null   bool  
 11  first_pos      5346 non-null   object
dtypes: bool(1), object(11)
memory usage: 464.8+ KB


Index(['desc', 'output', 'type', 'company', 'industry', 'url', 'alias',
       'desc_masked', 'output_masked', 'ent_dict', 'unsupported', 'first_pos'],
      dtype='object')

Extract relevant columns and handle missing values

In [64]:
df = df[['company','output','industry']]


Remove missing values

In [65]:
df = df.dropna()
print(df.head())

                      company  \
0            eftpos warehouse   
1                       welbi   
2                optinmonster   
3                    twine.fm   
4  mcb financial services ltd   

                                              output  \
0           Taking Care of Small Business Technology   
1              Build World-Class Recreation Programs   
2  Most Powerful Lead Generation Software for Mar...   
3              Hire quality freelancers for your job   
4                Financial Advisers Norwich, Norfolk   

                       industry  
0             computer hardware  
1  health, wellness and fitness  
2                      internet  
3                      internet  
4            financial services  


Drop missing values in rows

In [66]:
df = df.dropna()

df.isnull().sum()

,0
company,0
output,0
industry,0


Encoding industries

In [67]:
# Imported the LabelEncoder class from scikit-learn.
# Create a LabelEncoder object to convert industry names into integer labels.
# Fit the encoder to the 'industry' column and transform each industry
# into a unique numerical value, storing the result in a new column
# called 'industry_encoded'.
from sklearn.preprocessing import LabelEncoder

industry_encoder = LabelEncoder()

df['industry_encoded'] = industry_encoder.fit_transform(df['industry'])

Tokenise the slogans and business names in the dataset

In [68]:
# Load spaCy model for text processing
# Loading the small English spaCy language model for natural language processing tasks,
# such as tokenization and text preprocessing.
nlp = spacy.load("en_core_web_sm")

# Defining a function to preprocess slogan text by converting it to lowercase,
# tokenizing it, and removing punctuation.
def preprocess_text(text):
    # Converting all characters to lowercase to ensure consistency.
    text_lower = text.lower()
    # Processing the text using the spaCy language model.
    doc = nlp(text_lower)

    # Creating an empty list to store the cleaned tokens.
    processed_tokens = []

    # Iterating through each token in the processed document.
    for token in doc:
        # Keeping only tokens that are not punctuation marks.
        if not token.is_punct:
            processed_tokens.append(token.text)

    # Joining the cleaned tokens back into a single string and return it
    return " ".join(processed_tokens)

# Applying the preprocessing function to every slogan in the 'output' column
# and storing the cleaned text in a new column called 'processed_slogan'.
df["processed_slogan"] = df["output"].apply(preprocess_text)

# Displaying the first five rows of the DataFrame to verify the preprocessing results.
df.head()

,company,output,industry,industry_encoded,processed_slogan
0,eftpos warehouse,Taking Care of Small Business Technology,computer hardware,21,taking care of small business technology
1,welbi,Build World-Class Recreation Programs,"health, wellness and fitness",52,build world class recreation programs
2,optinmonster,Most Powerful Lead Generation Software for Mar...,internet,65,most powerful lead generation software for mar...
3,twine.fm,Hire quality freelancers for your job,internet,65,hire quality freelancers for your job
4,mcb financial services ltd,"Financial Advisers Norwich, Norfolk",financial services,41,financial advisers norwich norfolk


Tokenise the slogans

In [69]:
# Importing the Tokenizer class from TensorFlow Keras to convert text into
# numerical representations that can be used by a neural network.
from tensorflow.keras.preprocessing.text import Tokenizer

# Creating a Tokenizer object.
tokenizer = Tokenizer()

# Building the tokenizer's vocabulary by fitting it to the slogan text
# stored in the 'output' column of the DataFrame.
tokenizer.fit_on_texts(df['output'])

# Calculating the total number of unique words in the vocabulary.
# The '+1' accounts for the reserved index 0, which is used for padding.
total_words = len(tokenizer.word_index) + 1

# Displaying the total number of words in the tokenizer's vocabulary.
print(total_words)

6102


Create training sequences

In [70]:
# Creating an empty list to store the generated input sequences
# that will be used to train the slogan generation model.
input_sequences = []

# Loop through each slogan in the dataset.
for slogan in df['output']:
    # Converting the slogan into a sequence of integer tokens
    # based on the tokenizer's vocabulary.
    token_list = tokenizer.texts_to_sequences([slogan])[0]

    # Generate n-gram sequences of increasing length from the tokenized slogan.
    # For example, a token sequence [5, 8, 2, 9] becomes:
    # [5, 8], [5, 8, 2], [5, 8, 2, 9]
    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]

        # Add each n-gram sequence to the list of training sequences.
        input_sequences.append(n_gram)

Pad sequences

In [71]:
# Importing the pad_sequences function to ensure all input sequences have
# the same length, and import NumPy for numerical array operations.
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Determing the length of the longest input sequence.
# This value will be used to pad all shorter sequences to the same length.
max_sequence_len = max(len(seq) for seq in input_sequences)

# Padding all input sequences with zeros at the beginning ('pre' padding)
# so that every sequence has the same length, then convert the result
# into a NumPy array for efficient processing by the neural network.
input_sequences = np.array(
    pad_sequences(input_sequences,
                  maxlen=max_sequence_len,
                  padding='pre')
)

# Spliting the padded sequences into input features (X) and target labels (y).
# X contains all tokens except the last one, while y contains the final token
# that the model will learn to predict.
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

One-hot encoding output

In [72]:
# Importing the to_categorical function from TensorFlow Keras to convert
# integer class labels into one-hot encoded vectors.
from tensorflow.keras.utils import to_categorical

# Converting the target labels (y) into one-hot encoded format.
# Each label is represented as a binary vector with a length equal to
# the total number of unique words in the vocabulary. This format is
# required for training the neural network using multiclass classification.
y = to_categorical(y, num_classes=total_words)

## 2. Slogan Generator
- Build an LTSM model that can generate slogans based on an input industry

In [73]:
# Importing the required Keras layers and Model class for building the
# neural network used to generate slogans based on industry and text input.
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Concatenate, RepeatVector, Flatten
from tensorflow.keras.models import Model

# Defining the input layer for the encoded industry label.
industry_input = Input(shape=(1,), name="industry")

# Creating an embedding layer to learn a dense vector representation
# for each industry category.
industry_embedding = Embedding(
    input_dim=len(industry_encoder.classes_),
    output_dim=16
)(industry_input)

# Flatten the industry embedding from a 3D tensor to a 2D tensor
# so it can be repeated across all time steps.
industry_embedding = Flatten()(industry_embedding)

# Repeat the industry embedding so that it matches the length of
# the text input sequence for concatenation.
industry_embedding = RepeatVector(max_sequence_len-1)(industry_embedding)

# Defining the input layer for the tokenized text sequence.
text_input = Input(shape=(max_sequence_len-1,), name="text")

# Converting each word index in the input sequence into a dense vector
# representation using a word embedding layer.
text_embedding = Embedding(total_words, 100)(text_input)

# Processing the embedded text sequence using an LSTM layer and return
# the hidden state for each time step.
text_output = LSTM(128, return_sequences=True)(text_embedding)

# Concatenate the text features with the repeated industry embedding
# so that the model learns to generate slogans conditioned on the industry.
merged = Concatenate()([text_output, industry_embedding])

# Processing the combined features through a second LSTM layer to
# capture the overall sequence context.
merged = LSTM(128)(merged)

# Adding a fully connected output layer with a softmax activation
# to predict the next word from the entire vocabulary.
output = Dense(total_words, activation='softmax')(merged)

# Building the slogan generator model using both the industry and
# text inputs, with the predicted next word as the output.
generator = Model(inputs=[industry_input,text_input], outputs=output)

# Compiling the model using categorical cross-entropy loss for
# multiclass classification, the Adam optimizer, and accuracy
# as the evaluation metric.
generator.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Displaying a summary of the model architecture, including the
# layers, output shapes, and number of trainable parameters.
generator.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ industry            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text (InputLayer)   │ (None, 10)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 1, 16)     │      2,272 │ industry[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 10, 100)   │    610,200 │ text[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 16)        │          0 │ embedding_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ (None, 10, 128)   │    117,248 │ embedding_7[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_2     │ (None, 10, 16)    │          0 │ flatten_2[0][0]   │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 10, 144)   │          0 │ lstm_6[0][0],     │
│ (Concatenate)       │                   │            │ repeat_vector_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ (None, 128)       │    139,776 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 6102)      │    787,158 │ lstm_7[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,656,654 (6.32 MB)

 Trainable params: 1,656,654 (6.32 MB)

 Non-trainable params: 0 (0.00 B)

Preparing industry inputs

In [74]:
# Creating an empty list to store the encoded industry label
# corresponding to each generated training sequence.
industry_inputs = []

# Loop through each slogan and its associated encoded industry label.
for slogan, industry in zip(df['output'], df['industry_encoded']):

    # Converting the slogan into a sequence of integer tokens
    # using the fitted tokenizer.
    token_list = tokenizer.texts_to_sequences([slogan])[0]

    # For every n-gram sequence generated from the slogan,
    # append the same industry label so that each training
    # sequence has its corresponding industry information.
    for i in range(1, len(token_list)):
        industry_inputs.append(industry)

# Converting the list of industry labels into a NumPy array
# for efficient processing during model training.
industry_inputs = np.array(industry_inputs)

Train the model on the provided slogan dataset and test it by generating slogans for different industries

In [75]:
from tensorflow.keras.utils import to_categorical

# Ensure y is one-hot encoded for categorical_crossentropy

# Training the slogan generation model using both the encoded industry labels
# and the tokenized text sequences as inputs. The model learns to predict
# the next word in a slogan based on the previous words and the associated
# industry category.
history = generator.fit(
    [industry_inputs, X],
    y,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
573/573 ━━━━━━━━━━━━━━━━━━━━ 25s 39ms/step - accuracy: 0.0479 - loss: 7.5870 - val_accuracy: 0.0480 - val_loss: 7.5008
Epoch 2/50
573/573 ━━━━━━━━━━━━━━━━━━━━ 39s 36ms/step - accuracy: 0.0484 - loss: 7.0737 - val_accuracy: 0.0458 - val_loss: 7.5858
Epoch 3/50
573/573 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step - accuracy: 0.0510 - loss: 6.9069 - val_accuracy: 0.0493 - val_loss: 7.6650
Epoch 4/50
573/573 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step - accuracy: 0.0610 - loss: 6.6998 - val_accuracy: 0.0534 - val_loss: 7.7289
Epoch 5/50
573/573 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step - accuracy: 0.0677 - loss: 6.4855 - val_accuracy: 0.0534 - val_loss: 7.7735
Epoch 6/50
573/573 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step - accuracy: 0.0765 - loss: 6.2766 - val_accuracy: 0.0593 - val_loss: 7.8898
Epoch 7/50
573/573 ━━━━━━━━━━━━━━━━━━━━ 23s 40ms/step - accuracy: 0.0847 - loss: 6.0772 - val_accuracy: 0.0596 - val_loss: 7.9188
Epoch 8/50
573/573 ━━━━━━━━━━━━━━━━━━━━ 40s 38ms/step - accuracy: 0.0890 - loss: 5.8877 - 

Test the model by generating slogans for different industries

In [76]:
import numpy as np

def generate_slogan(industry_name, seed_text="", next_words=6):
    """
    Generate a business slogan using the trained LSTM slogan generator.

    The function generates a slogan by predicting one word at a time based
    on a specified industry and an optional seed text. The generated word is
    appended to the input text and used as the input for the next prediction
    until the desired number of words has been produced.

    Args:
        industry_name (str):
            The name of the industry for which a slogan will be generated.
            The industry must exist in the fitted LabelEncoder.

        seed_text (str, optional):
            The starting text for slogan generation. If no seed text is
            provided, the model begins generating from an empty sequence.
            Defaults to "".

        next_words (int, optional):
            The number of words to generate after the seed text.
            Defaults to 6.

    Returns:
        str:
            The complete generated slogan consisting of the seed text
            followed by the predicted words.
    """

    # Converting the industry name into its corresponding numerical label
    # so it can be used as input to the neural network.
    industry = industry_encoder.transform([industry_name])[0]

    # Generating the specified number of words one at a time.
    for _ in range(next_words):

        # Convert the current slogan text into a sequence of integer
        # tokens using the fitted tokenizer.
        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        # Pad the sequence with leading zeros so that its length
        # matches the input size expected by the model.
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len-1,
            padding='pre'
        )

        # Predicting the probability distribution of the next word using
        # both the encoded industry and the current text sequence.
        predicted = generator.predict(
            [np.array([[industry]]), token_list],
            verbose=0
        )

        # Selecting the word with the highest predicted probability.
        predicted_word = np.argmax(predicted)

        # Initialise a variable to store the predicted word.
        output_word = ""

        # Finding the word that corresponds to the predicted token index.
        for word,index in tokenizer.word_index.items():

            if index == predicted_word:
                output_word = word
                break

        # Appending the predicted word to the slogan so it can be used
        # as input for the next prediction.
        seed_text += " " + output_word

     # Return the completed generated slogan.
    return seed_text

Compiling the model

## 3. Slogan Classifer
- Create a classification model that predicts the industry of a business based on its slogan.

In [77]:
# Reuse the existing DataFrame and tokenizer from Part 1 instead of reloading the dataset.
# Redundant preprocessing removed.


Pad them

In [78]:
# Importing the pad_sequences function to ensure that all tokenized slogan
# sequences have the same length before being used by the classifier.
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder # Ensure LabelEncoder is available

# Re-add the 'industry_encoded' column to the df, using the existing industry_encoder.
# This assumes industry_encoder is already defined and fitted from cell eNidBdznWTaS.
# The df was overwritten in cell xEI7kJhRhk3u, losing this column.
industry_encoder = LabelEncoder() # Re-initialize and fit for this specific df context.
df['industry_encoded'] = industry_encoder.fit_transform(df['industry'])

# Tokenize the slogans for the classifier input
X_classifier = tokenizer.texts_to_sequences(df['output'])

# Determine the length of the longest slogan sequence in the dataset.
# This value will be used as the maximum sequence length for padding.
max_len = max(len(i) for i in X_classifier)

# Padding all slogan sequences with zeros at the end ('post' padding)
# so that every input sequence has the same length. This ensures the
# data has a consistent shape for training the neural network.
X_classifier = pad_sequences(
    X_classifier,
    maxlen=max_len,
    padding='post'
)

# Defining the target labels for the classification model.
# The encoded industry values represent the class that the model
# will learn to predict from each slogan.
y_classifier = df['industry_encoded']

Split the dataset into training and testing sets.

In [79]:
# Importing the train_test_split function from scikit-learn to divide the
# dataset into separate training and testing sets.
from sklearn.model_selection import train_test_split

# Split the classifier dataset into training and testing subsets.
# - X_classifier contains the padded slogan sequences (input features).
# - y_classifier contains the encoded industry labels (target values).
# - test_size=0.2 reserves 20% of the data for testing and uses the
#   remaining 80% for training.
# - random_state=42 ensures that the split is reproducible each time
#   the code is executed.
X_train, X_test, y_train, y_test = train_test_split(
    X_classifier,
    y_classifier,
    test_size=0.2,
    random_state=42
)

Building the classifier

In [80]:
# Import the Sequential model and the neural network layers required
# to build the LSTM-based slogan classification model.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Creating a Sequential model in which layers are added one after another.
classifier = Sequential()

# Adding an embedding layer to convert each word index in the input
# sequence into a dense vector representation. This allows the model
# to learn meaningful relationships between words.
classifier.add(
    Embedding(
        input_dim=len(tokenizer.word_index)+1,
        output_dim=128,
        input_length=max_len
    )
)

# Adding an LSTM layer to learn sequential patterns and contextual
# information from the slogan text.
classifier.add(LSTM(128))

# Adding a fully connected (Dense) hidden layer with the ReLU activation
# function to learn higher-level features extracted by the LSTM layer.
classifier.add(Dense(64,activation='relu'))

# Adding the output layer with one neuron for each industry class.
# The softmax activation function produces the probability that
# a slogan belongs to each industry.
classifier.add(Dense(
    len(industry_encoder.classes_),
    activation='softmax'
))

# Compile the classification model.
# - Adam is used as the optimizer to efficiently update the model weights.
# - Sparse categorical cross-entropy is used because the target labels
#   are integer-encoded rather than one-hot encoded.
# - Accuracy is used to evaluate the model's classification performance.
classifier.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display a summary of the classifier architecture, including the
# layers, output shapes, and number of trainable parameters.
classifier.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Train the model

In [81]:
# Train the slogan classification model using the training dataset.
# The model learns to predict the correct industry based on the
# tokenized and padded slogan text.
classifier.fit(
    X_train,
    y_train,
    epochs=5,  # Train the model for 5 complete passes through the training data.
    validation_split=0.2  # Reserve 20% of the training data to evaluate the model's
                         # performance during training and monitor for overfitting.
)

Epoch 1/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.0658 - loss: 4.5005 - val_accuracy: 0.0829 - val_loss: 4.2885
Epoch 2/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.0851 - loss: 4.1284 - val_accuracy: 0.0806 - val_loss: 4.2094
Epoch 3/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.0868 - loss: 3.9055 - val_accuracy: 0.0818 - val_loss: 4.2467
Epoch 4/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.0985 - loss: 3.7286 - val_accuracy: 0.0701 - val_loss: 4.2520
Epoch 5/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.1287 - loss: 3.4913 - val_accuracy: 0.1005 - val_loss: 4.3493


Evaluate how well the model performs

In [82]:
# Evaluate the trained classification model using the test dataset.
# The model returns the loss value and the classification accuracy,
# which measure how well it performs on previously unseen data.
loss, accuracy = classifier.evaluate(X_test, y_test)

# Displaying the model's classification accuracy on the test dataset.
# Accuracy represents the proportion of correctly classified slogans.
print("Accuracy:", accuracy)

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0925 - loss: 4.2891
Accuracy: 0.09252336621284485


### 4. Combining both models
- Use the generator to create new slogans for a given industry

In [83]:
# Reuse the existing DataFrame and tokenizer from Part 1 instead of reloading the dataset.
# Redundant preprocessing removed.


Pass the generated slogans through the classifier to see if it can correctly predict the industry

In [84]:
# Generate a new slogan using the previously defined function
new_slogan = generate_slogan('computer hardware', seed_text='future of', next_words=5)
print(f"Generated Slogan: {new_slogan}")

# Convert the generated slogan into a sequence of integer tokens
# using the tokenizer that was trained for the classification model.
sequence = tokenizer.texts_to_sequences([new_slogan])

# Padding the token sequence with zeros at the end so that it matches
# the input length expected by the classifier.
sequence = pad_sequences(sequence,maxlen=max_len,padding='post')

# Using the trained classifier to predict the probability of the
# slogan belonging to each industry category.
prediction = classifier.predict(sequence)

# Identifying the industry with the highest predicted probability
# and convert the encoded label back to its original industry name.
predicted = industry_encoder.inverse_transform(
    [np.argmax(prediction)]
)

# Display the predicted industry for the generated slogan.
print(f"Predicted Industry: {predicted[0]}")

Generated Slogan: future of innovation reviews blog today in
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
Predicted Industry: publishing


### Comparison of the Generated Slogans and the Classifier's Predictions

After training the slogan generation model for **50 epochs**, the quality of the generated slogans improved compared to the initial outputs produced before training. The generated slogans contained more meaningful words, showed better sentence structure, and were more closely related to the intended industry, although some slogans still contained repetitive words or lacked complete grammatical correctness.

The classifier generally performed better than the slogan generator. While the generator attempted to create new slogans by predicting the next word in a sequence, the classifier analysed the completed slogan and predicted the most likely industry based on the learned patterns in the training data. In most cases, the classifier was able to assign an industry label that was consistent with the overall theme of the generated slogan, even when the slogan was not perfectly written.

The main difference between the two models is their objectives. The slogan generator is a text generation model that focuses on producing coherent and creative sequences of words. As a result, it is more susceptible to repetition and grammatical errors. The classifier, however, is a supervised classification model whose objective is to recognise patterns associated with industry categories rather than produce natural language. Consequently, it can correctly identify an industry even when the generated slogan is only partially coherent.

Training the generator for 50 epochs resulted in noticeable improvements over the earlier untrained model, which repeatedly generated the same word. The longer training period enabled the model to learn more meaningful word relationships and produce slogans that were more relevant to their industries. However, the generated slogans were still not as fluent or diverse as professionally written slogans, indicating that additional improvements such as more training data, hyperparameter tuning, regularisation, or improved text generation techniques could further enhance performance.

Overall, the results demonstrate that the classifier achieved more reliable performance than the slogan generator. The classifier consistently recognised industry-related patterns, whereas the generator, although improved after 50 epochs, still had some difficulty producing completely natural and grammatically correct slogans.
